## Example 1: A simple LightningModule

Define a LightningModule and inspect it.

In [ ]:
# Example 1: Simple LightningModule Definition
import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

class SimpleMLP(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(4, 3)
        
    def forward(self, x):
        return self.layer(x)
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y)
        self.log('train_loss', loss)
        return loss
        
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters())

# Create model and sample data
model = SimpleMLP()
X = torch.randn(100, 4)
y = torch.randint(0, 3, (100,))
print(f'Model created: {type(model).__name__}')
print(f'Model parameters: {sum(p.numel() for p in model.parameters())}')
print('Example 1: Simple Lightning module created successfully')

## Example 2: Training with Lightning Trainer

Demonstrate basic training loop with Lightning Trainer.

In [ ]:
# Example 2: Training with Lightning Trainer
# Create simple dataset
X_train = torch.randn(200, 4)
y_train = torch.randint(0, 3, (200,))
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Create model
class TrainingMLP(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(4, 3)
        
    def forward(self, x):
        return self.layer(x)
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y)
        return loss
        
    def configure_optimizers(self):
        return torch.optim.SGD(self.parameters(), lr=0.01)

model = TrainingMLP()
trainer = L.Trainer(max_epochs=2, enable_progress_bar=False, logger=False)
trainer.fit(model, train_loader)
print('Example 2: Model training completed successfully')

## Example 3: Lightning DataModule

Demonstrate data organization with LightningDataModule.

In [ ]:
# Example 3: Lightning DataModule
class SimpleDataModule(L.LightningDataModule):
    def __init__(self, batch_size=32):
        super().__init__()
        self.batch_size = batch_size
        
    def prepare_data(self):
        # Download or prepare data (runs once)
        pass
        
    def setup(self, stage=None):
        # Create datasets for each split
        X = torch.randn(300, 4)
        y = torch.randint(0, 3, (300,))
        
        if stage == 'fit' or stage is None:
            self.train_dataset = TensorDataset(X[:200], y[:200])
            self.val_dataset = TensorDataset(X[200:250], y[200:250])
            
        if stage == 'test' or stage is None:
            self.test_dataset = TensorDataset(X[250:], y[250:])
            
    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)
        
    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

dm = SimpleDataModule(batch_size=16)
dm.setup('fit')
print(f'DataModule created with batch size: {dm.batch_size}')
print(f'Train dataset size: {len(dm.train_dataset)}')
print('Example 3: DataModule setup completed successfully')

## Example 4: Callbacks and Logging

Demonstrate Lightning callbacks and logging functionality.

In [ ]:
# Example 4: Callbacks and logging
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

class LoggingMLP(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(4, 3)
        
    def forward(self, x):
        return self.layer(x)
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y)
        acc = (y_hat.argmax(dim=1) == y).float().mean()
        self.log('train_loss', loss)
        self.log('train_acc', acc)
        return loss
        
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters())

# Setup callbacks
early_stop = EarlyStopping(monitor='train_loss', patience=3, mode='min')
checkpoint = ModelCheckpoint(monitor='train_loss', mode='min')

model = LoggingMLP()
trainer = L.Trainer(max_epochs=2, callbacks=[early_stop, checkpoint], 
                   enable_progress_bar=False, logger=False)
trainer.fit(model, train_loader)
print('Example 4: Training with callbacks completed successfully')

## Example 5: Validation and Testing

Demonstrate validation loop and model testing with Lightning.

In [ ]:
# Example 5: Validation and testing
class ValidatedMLP(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(4, 3)
        
    def forward(self, x):
        return self.layer(x)
        
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y)
        return loss
        
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        val_loss = F.cross_entropy(y_hat, y)
        val_acc = (y_hat.argmax(dim=1) == y).float().mean()
        self.log('val_loss', val_loss)
        self.log('val_acc', val_acc)
        
    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        test_acc = (y_hat.argmax(dim=1) == y).float().mean()
        self.log('test_acc', test_acc)
        
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters())

# Create validation data
X_val, y_val = torch.randn(50, 4), torch.randint(0, 3, (50,))
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=16)

model = ValidatedMLP()
trainer = L.Trainer(max_epochs=1, enable_progress_bar=False, logger=False)
trainer.fit(model, train_loader, val_loader)
trainer.test(model, val_loader)
print('Example 5: Validation and testing completed successfully')